# 02 — DuckDB Database Setup

## Objective

This notebook creates the DuckDB database used for the project.

It loads:
- The consolidated IDS dataset from Parquet into `ids_staging`.
- The drug classification reference into `drug_reference`.

These tables are the inputs for the SQL scripts that construct the
analytical star schema.

In [1]:
# Imports and setup
from pathlib import Path
import duckdb
import pandas as pd

print("DuckDB version:", duckdb.__version__)

DuckDB version: 2.0.0


In [2]:
## If DuckDB is not installed, uncomment and run this once:
# %pip install duckdb

In [3]:
# Project paths
current_path = Path.cwd()

project_root = next(
    (
        path for path in [current_path, *current_path.parents]
        if (path / "data" / "processed").is_dir()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root containing data/processed."
    )

processed_dir = project_root / "data" / "processed"

# The existing filename is retained here.
PARQUET_FILENAME = "ids_2011_2026.parquet"

parquet_path = processed_dir / PARQUET_FILENAME
reference_path = processed_dir / "drug_reference.csv"
db_path = processed_dir / "seizure_signals.duckdb"

for path in [parquet_path, reference_path]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

print("Project root:", project_root)
print("Parquet file:", parquet_path.name)
print("Drug reference:", reference_path.name)
print("Database:", db_path.name)

Project root: C:\Users\joaqu\OneDrive\Escritorio\CV\Postulaciones\XX. UNIDO\Personal Project\GitHub\seizure-signals
Parquet file: ids_2011_2026.parquet
Drug reference: drug_reference.csv
Database: seizure_signals.duckdb


In [4]:
# Connect to DuckDB
con = duckdb.connect(str(db_path))

print("Connected to DuckDB.")

Connected to DuckDB.


In [5]:
# Load the IDS dataset
con.execute("""
    CREATE OR REPLACE TABLE ids_staging AS
    SELECT *
    FROM read_parquet(?)
""", [str(parquet_path)])

print("ids_staging created successfully.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ids_staging created successfully.


In [6]:
# Load the drug reference
con.execute("""
    CREATE OR REPLACE TABLE drug_reference AS
    SELECT *
    FROM read_csv_auto(?)
""", [str(reference_path)])

print("drug_reference created successfully.")

drug_reference created successfully.


In [7]:
# Validate the tables
table_counts = con.execute("""
    SELECT 'ids_staging' AS table_name, COUNT(*) AS total_records
    FROM ids_staging

    UNION ALL

    SELECT 'drug_reference', COUNT(*)
    FROM drug_reference
""").df()

display(table_counts)

,table_name,total_records
0,ids_staging,2841288
1,drug_reference,184


Expected results:
- ids_staging: 2,841,288 records.
- drug_reference: 184 records.

In [8]:
# Inspect the staging table
display(
    con.execute("DESCRIBE ids_staging").df()
)

,column_name,column_type,null,key,default,extra
0,Seizure Date,TIMESTAMP_NS,YES,None,None,None
1,Country/Territory of Seizure,VARCHAR,YES,None,None,None
2,ISO3,VARCHAR,YES,None,None,None
3,Subregion,VARCHAR,YES,None,None,None
4,Region,VARCHAR,YES,None,None,None
5,Drug/Substance,VARCHAR,YES,None,None,None
6,Quantity Seized,DOUBLE,YES,None,None,None
7,Measurement Unit,VARCHAR,YES,None,None,None
8,City,VARCHAR,YES,None,None,None
9,Administrative Region,VARCHAR,YES,None,None,None


In [9]:
# Check date coverage and annual counts
date_summary = con.execute("""
    SELECT
        MIN(CAST("Seizure Date" AS DATE)) AS first_date,
        MAX(CAST("Seizure Date" AS DATE)) AS last_date,
        COUNT(*) AS total_records
    FROM ids_staging
""").df()

display(date_summary)

,first_date,last_date,total_records
0,2011-01-01,2026-06-30,2841288


In [11]:
annual_counts = con.execute("""
    SELECT
        source_year,
        COUNT(*) AS total_records,
        COUNT(DISTINCT ISO3) AS reporting_countries,
        COUNT(DISTINCT "Drug/Substance") AS distinct_drugs
    FROM ids_staging
    GROUP BY source_year
    ORDER BY source_year
""").df()

display(annual_counts)


,source_year,total_records,reporting_countries,distinct_drugs
0,2011,41806,67,101
1,2012,62090,72,143
2,2013,62036,63,139
3,2014,265457,53,175
4,2015,238294,45,140
5,2016,307817,46,145
6,2017,232409,54,107
7,2018,220003,82,112
8,2019,119368,77,106
9,2020,151682,84,120


In [12]:
# Check the drug reference
reference_summary = con.execute("""
    SELECT
        COUNT(*) AS total_records,
        COUNT(DISTINCT "Drug Name/Type") AS unique_drugs,
        COUNT("Drug class/ group") AS classified_records,
        COUNT(*) - COUNT("Drug class/ group") AS missing_classes
    FROM drug_reference
""").df()

display(reference_summary)

,total_records,unique_drugs,classified_records,missing_classes
0,184,184,184,0


In [13]:
# Close the connection
con.close()

print("DuckDB connection closed.")

DuckDB connection closed.


## Conclusions

- The consolidated IDS dataset was successfully loaded into DuckDB.
- The staging table preserves the original seizure-level records and source metadata.
- The drug reference was loaded as a separate table to support drug classification.
- Record counts, date coverage, and table structures were checked after ingestion.
- The resulting database is ready for star-schema construction using SQL.